# ASR Russian Numbers — Kaggle Inference Notebook

This notebook:
1. Installs dependencies
2. Downloads model source code from GitHub
3. Downloads trained weights from a GitHub Release
4. Runs inference on the test set
5. Saves `submission.csv`

In [ ]:
# ── 0. Configuration — UPDATE these before running ───────────────────────────
GITHUB_REPO   = "YOUR_USERNAME/asr-russian-numbers"   # e.g. "alexcodeaap/asr-russian-numbers"
RELEASE_TAG   = "v1.0.0"                               # GitHub release tag with best.pt
WEIGHTS_FILE  = "best.pt"                              # filename inside the release

TEST_CSV      = "/kaggle/input/YOUR_DATASET/test.csv"  # Kaggle input path
AUDIO_DIR     = "/kaggle/input/YOUR_DATASET/"          # root dir for audio files
OUTPUT_CSV    = "/kaggle/working/submission.csv"

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

run("pip install -q num2words editdistance")
print("Dependencies installed.")

In [ ]:
# ── 2. Clone source code from GitHub ─────────────────────────────────────────
import os

REPO_DIR = "/kaggle/working/asr-russian-numbers"

if os.path.exists(REPO_DIR):
    run(f"rm -rf {REPO_DIR}")

run(f"git clone --depth 1 https://github.com/{GITHUB_REPO}.git {REPO_DIR}")
sys.path.insert(0, REPO_DIR)
print(f"Cloned into {REPO_DIR}")

In [ ]:
# ── 3. Download model weights from GitHub Release ────────────────────────────
import urllib.request

WEIGHTS_URL = (
    f"https://github.com/{GITHUB_REPO}/releases/download/{RELEASE_TAG}/{WEIGHTS_FILE}"
)
WEIGHTS_PATH = f"/kaggle/working/{WEIGHTS_FILE}"

print(f"Downloading weights from:\n  {WEIGHTS_URL}")
urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)

size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
print(f"Downloaded {WEIGHTS_FILE} ({size_mb:.1f} MB)")

In [ ]:
# ── 4. Load config, model, and vocabulary ────────────────────────────────────
import yaml
import torch
from src.model.ctc_model import build_model
from src.text.vocabulary import Vocabulary

CONFIG_PATH = os.path.join(REPO_DIR, "config.yaml")
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

vocab = Vocabulary()
model = build_model(cfg, vocab.size).to(device)

ckpt = torch.load(WEIGHTS_PATH, map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()

epoch = ckpt.get("epoch", "?")
best_cer = ckpt.get("best_cer", "?")
print(f"Loaded checkpoint: epoch={epoch}, best_HM-CER={best_cer}")
print(f"Model parameters: {model.count_parameters():,}")

In [ ]:
# ── 5. Build test data loader ─────────────────────────────────────────────────
from torch.utils.data import DataLoader
from src.data.dataset import TestDataset, collate_fn_test

test_ds = TestDataset(TEST_CSV, AUDIO_DIR, cfg)
test_loader = DataLoader(
    test_ds,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn_test,
)
print(f"Test samples: {len(test_ds)}")

In [ ]:
# ── 6. Run inference ──────────────────────────────────────────────────────────
from tqdm.notebook import tqdm
from src.utils.decoder import batch_decode
from src.text.normalization import words_to_num

filenames_all, predictions_all = [], []

with torch.no_grad():
    for mels, mel_lengths, filenames in tqdm(test_loader, desc="Inference"):
        mels = mels.to(device)
        mel_lengths = mel_lengths.to(device)

        log_probs, out_lengths = model(mels, mel_lengths)
        hyps_words = batch_decode(log_probs, out_lengths, vocab)

        for fname, words in zip(filenames, hyps_words):
            digit_str = words_to_num(words)
            filenames_all.append(fname)
            predictions_all.append(digit_str)

print(f"Inference complete. {len(predictions_all)} predictions.")

In [ ]:
# ── 7. Save submission CSV ────────────────────────────────────────────────────
import pandas as pd

submission = pd.DataFrame({
    "filename": filenames_all,
    "transcription": predictions_all,
})
submission.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(submission)} rows → {OUTPUT_CSV}")
submission.head(10)

In [ ]:
# ── 8. Quick sanity check ─────────────────────────────────────────────────────
print("Unique predictions:", submission["transcription"].nunique())
print("Sample predictions:")
print(submission.sample(min(10, len(submission))).to_string(index=False))

# Warn if many predictions are non-numeric (parsing failures)
non_numeric = submission[~submission["transcription"].str.isnumeric()]
if len(non_numeric) > 0:
    pct = 100 * len(non_numeric) / len(submission)
    print(f"\nWARNING: {len(non_numeric)} ({pct:.1f}%) predictions are non-numeric")
    print(non_numeric.head())
else:
    print("\nAll predictions are numeric strings. Ready to submit!")